# SVillage 再現ノートブック：予想外のインフレと政府債務／円安で得をするのは誰か

SVillage（https://s-v.jp.net）の前編・後編の記事で使った数値と図を、すべて再現するノートブックです。

- 前編：予想外のインフレで誰が得をする？――政府債務と「見えない税」（図1〜5）
- 後編：円安で得をするのは誰か――政府・企業・家計の海外資産（図1〜2、家計の金融資産の数値）

**使い方**

1. 同じフォルダにある `svillage_unexpected_inflation_colab_package.zip` を用意します。
2. 下の「0. 準備」のセルを実行すると、zipのアップロードを求められます。選ぶと `data/` に展開されます。
3. あとは上から順に実行してください。

データはすべて、2026年9月30日に一次ソースから取得したものです。PDFやExcelの一次データは、整形済みのCSVにして同梱しています。各セルのコメントに、取得元のURLを書いてあります。FREDの2系列（外貨準備・円ドル相場）は、`REFETCH_FRED = True` にすると最新値を取り直せます。

## 0. 準備

In [ ]:
# 日本語フォントとデータの準備（Colab用）
import os, sys, zipfile
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip -q install japanize-matplotlib
    if not os.path.exists("data/01_mitoshi_cpi_forecast.csv"):
        from google.colab import files
        up = files.upload()  # svillage_unexpected_inflation_colab_package.zip を選ぶ
        for name in up:
            zipfile.ZipFile(name).extractall(".")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
try:
    import japanize_matplotlib  # noqa: F401
except ImportError:
    plt.rcParams["font.family"] = ["Yu Gothic", "Meiryo", "Noto Sans CJK JP", "IPAexGothic"]
plt.rcParams["axes.unicode_minus"] = False
NAVY, ORANGE, GRAY, RED, GREEN = "#1f4e8c", "#d9622b", "#8a8a8a", "#c0392b", "#2e8b57"
DATA = "data"
REFETCH_FRED = False  # True にすると FRED から外貨準備・円ドル相場を取り直す

## 1. データの読み込み

In [ ]:
# 1) 内閣府「政府経済見通し」各年度版（年度開始前の閣議決定版）の消費者物価（総合）見通し（%）
#    https://www5.cao.go.jp/keizai1/mitoshi/mitoshikako.html （各年度のPDF本文から読み取り）
mitoshi = pd.read_csv(f"{DATA}/01_mitoshi_cpi_forecast.csv", index_col="fy")["cpi_forecast_pct"]

# 2) 総務省「2025年基準消費者物価指数」長期時系列 全国 総合（月次）
#    https://www.stat.go.jp/data/cpi/1.html （e-Stat statInfId=000040482943）
cpi_m = pd.read_csv(f"{DATA}/02_cpi_monthly.csv", dtype={"ym": str})
cpi_m["y"], cpi_m["m"] = cpi_m["ym"].str[:4].astype(int), cpi_m["ym"].str[4:].astype(int)
cpi_m["fy"] = cpi_m["y"] - (cpi_m["m"] <= 3)
g = cpi_m.groupby("fy")["cpi_all_items_2025base"].agg(["mean", "count"])
cpi_fy = g[g["count"] == 12]["mean"]                 # 年度平均
infl_fy = (cpi_fy / cpi_fy.shift(1) - 1) * 100       # 前年度比（%）

# 3) 財務省「最近20カ年間の年度末の国債残高の推移」普通国債 計（兆円）
#    https://www.mof.go.jp/jgbs/reference/appendix/zandaka01.pdf
bal = pd.read_csv(f"{DATA}/03_jgb_general_bonds_balance.csv", index_col="fy")["general_bonds_trillion_yen"]

# 4) 財務省「普通国債の利率加重平均の各年ごとの推移」（年度末、%）
#    https://www.mof.go.jp/jgbs/reference/appendix/zandaka05.pdf
coupon = pd.read_csv(f"{DATA}/04_jgb_avg_coupon.csv", index_col="fy")["avg_coupon_pct"]

# 5) 内閣府「2026年4-6月期2次速報」名目年度 国内総生産（兆円）
#    https://www.esri.cao.go.jp/jp/sna/data/data_list/sokuhou/files/2026/qe262_2/tables/gaku-mfy2622.csv
gdp = pd.read_csv(f"{DATA}/05_nominal_gdp_fy.csv", index_col="fy")["nominal_gdp_trillion_yen"]

# 6) 財務省「昭和54年度以降の税収の推移」一般会計税収（兆円、決算。2025年度は決算概数）
#    https://www.mof.go.jp/tax_policy/summary/condition/zeisyu.xls
tax = pd.read_csv(f"{DATA}/06_tax_revenue_fy.csv", index_col="fy")["general_account_tax_trillion_yen"]

# 7) 厚生労働省「毎月勤労統計調査」長期時系列 指数（2020年=100）調査産業計・5人以上・就業形態計、暦年
#    https://www.e-stat.go.jp/stat-search/files?page=1&toukei=00450071&tstat=000001011791
w = pd.read_csv(f"{DATA}/07_wages_index_cy.csv", index_col="year")
w["price_index"] = w["nominal_wage_index"] / w["real_wage_index"] * 100

# 8) 財務省「国際収支状況」第一次所得収支の推移（暦年、兆円、ネット）
#    https://www.mof.go.jp/policy/international_policy/reference/balance_of_payments/bp_trend/bpnet/sbp/s-3/6s-3-1.csv
pi = pd.read_csv(f"{DATA}/08_primary_income_cy.csv", index_col="year")

# 9) 外貨準備（金を除く、百万ドル）FRED: TRESEGJPM052N、円ドル（月平均）FRED: EXJPUS
if REFETCH_FRED:
    fr = lambda s: pd.read_csv(f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={s}", parse_dates=[0], index_col=0).iloc[:, 0]
    rs = pd.concat([fr("TRESEGJPM052N"), fr("EXJPUS")], axis=1, keys=["reserves_ex_gold_usd_mn", "jpy_per_usd"]).dropna()
else:
    rs = pd.read_csv(f"{DATA}/09_fx_reserves_usdjpy_monthly.csv", parse_dates=["date"], index_col="date")
rs["usd_tn"] = rs["reserves_ex_gold_usd_mn"] / 1e6
rs["jpy_tn"] = rs["reserves_ex_gold_usd_mn"] * rs["jpy_per_usd"] / 1e6

# 10) 日本銀行「資金循環統計」家計（四半期末、億円）  https://www.stat-search.boj.or.jp/info/fof2_jp.zip
fof = pd.read_csv(f"{DATA}/10_fof_household_quarterly_oku_yen.csv", dtype={"quarter_yyyyqq": str}, index_col="quarter_yyyyqq")
print("読み込み完了")

## 2. 記事の数値を再計算する

In [ ]:
yrs = list(range(2003, 2026))
act = infl_fy.reindex(yrs).round(1)
exp = mitoshi.reindex(yrs)
surprise = act - exp
b = bal.loc[2007:2025]
real = b / cpi_fy.reindex(b.index) * cpi_fy[2007]        # 2007年度の物価で換算
ratio = b / gdp.reindex(b.index) * 100
growth = (gdp / gdp.shift(1) - 1) * 100
q = fof.index[-1]
hh = fof.loc[q] / 1e4                                      # 億円→兆円

rows = [
    ("前編", "予想外インフレ FY2022〜2025（%pt）", ", ".join(f"{y}:{surprise[y]:+.1f}" for y in (2022, 2023, 2024, 2025))),
    ("前編", "普通国債残高 FY2025（兆円）", f"{b[2025]:,.0f}"),
    ("前編", "同 2007年度の物価で換算（兆円）", f"{real[2025]:,.0f}"),
    ("前編", "普通国債残高/名目GDP FY2022→FY2025（%）", f"{ratio[2022]:.0f} → {ratio[2025]:.0f}"),
    ("前編", "名目GDP成長率 FY2022〜25の平均（%）", f"{growth.loc[2022:2025].mean():.2f}"),
    ("前編", "普通国債の平均利率 FY2022〜25末（%）", ", ".join(f"{coupon[y]:.2f}" for y in range(2022, 2026))),
    ("前編", "名目賃金・物価・実質賃金 2025（2020=100）", f"{w.loc[2025,'nominal_wage_index']:.1f} / {w.loc[2025,'price_index']:.1f} / {w.loc[2025,'real_wage_index']:.1f}"),
    ("前編", "税収 FY2025（兆円）", f"{tax[2025]:.1f}"),
    ("前編", "FY2020→2025 税収・名目GDPの増加率（%）", f"{(tax[2025]/tax[2020]-1)*100:.0f} / {(gdp[2025]/gdp[2020]-1)*100:.0f}"),
    ("後編", "第一次所得収支・直接投資収益・再投資収益 2025（兆円）", f"{pi.loc[2025,'primary_income']:.1f} / {pi.loc[2025,'direct_investment_income']:.1f} / {pi.loc[2025,'reinvested_earnings']:.1f}"),
    ("後編", "外貨準備の円換算 2012年1月→直近（兆円）", f"{rs['jpy_tn'].loc['2012-01-01']:.0f} → {rs['jpy_tn'].iloc[-1]:.0f}（{rs.index[-1]:%Y年%m月}）"),
    ("後編", f"家計の金融資産（{q[:4]}年第{int(q[4:])}四半期末、兆円）", f"{hh['total_financial_assets']:,.0f}"),
    ("後編", "うち現金・預金の割合（%）", f"{hh['currency_and_deposits']/hh['total_financial_assets']*100:.1f}"),
    ("後編", "外貨預金＋対外証券投資（兆円、割合%）", f"{hh['foreign_currency_deposits']+hh['outward_portfolio_investment']:.1f}（{(hh['foreign_currency_deposits']+hh['outward_portfolio_investment'])/hh['total_financial_assets']*100:.1f}%）"),
]
pd.set_option("display.max_colwidth", 80)
pd.DataFrame(rows, columns=["記事", "項目", "値"])

## 3. 図を再現する

記事の図と同じデータで描きます（体裁は記事の図を簡略化しています）。

In [ ]:
# 前編 図1　政府の物価見通しと実際の物価（年度）
fig, (ax, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True, gridspec_kw={"height_ratios": [3, 2]})
ax.plot(yrs, exp.values, color=GRAY, ls="--", marker="o", label="政府経済見通し（年度開始前の予想）")
ax.plot(yrs, act.values, color=NAVY, lw=2.5, marker="o", label="実際の消費者物価上昇率")
ax.axhline(0, color="#999", lw=0.8); ax.set_ylabel("前年度比（%）"); ax.legend(frameon=False)
ax.set_title("図1　政府の物価見通しと実際の物価（年度）", loc="left")
ax2.bar(yrs, surprise.values, color=[RED if v > 0 else "#9bb3d1" for v in surprise.values])
ax2.axhline(0, color="#999", lw=0.8); ax2.set_ylabel("実績－予想（%pt）"); ax2.set_title("予想外インフレ", loc="left")
plt.tight_layout(); plt.show()

In [ ]:
# 前編 図2　普通国債残高：額面・物価で割り引いた額・対GDP比
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(11, 5))
ax.plot(b.index, b.values, color=NAVY, lw=2.5, label="額面（名目）")
ax.plot(real.index, real.values, color=ORANGE, lw=2.5, label="物価で割り引いた額（2007年度の物価）")
ax.set_ylabel("兆円"); ax.legend(frameon=False); ax.set_title("普通国債残高（年度末）", loc="left")
ax2.plot(ratio.index, ratio.values, color=RED, lw=2.5); ax2.set_title("普通国債残高÷名目GDP（%）", loc="left")
fig.suptitle("図2　国の借金：額面は増え続けるが、物価で割り引くと2022年度から減っている", x=0.01, ha="left")
plt.tight_layout(); plt.show()

In [ ]:
# 前編 図3　経済の伸びと国債の平均金利
y4 = list(range(2007, 2026))
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(y4, growth.reindex(y4), color=NAVY, lw=2.5, marker="o", label="名目GDP成長率")
ax.plot(y4, coupon.reindex(y4), color=RED, lw=2.5, label="普通国債の平均利率（年度末）")
ax.plot(y4, infl_fy.reindex(y4), color=GRAY, ls="--", label="消費者物価上昇率")
ax.axhline(0, color="#999", lw=0.8); ax.set_ylabel("%"); ax.legend(frameon=False)
ax.set_title("図3　経済の伸びが、国債の平均金利を上回っている", loc="left")
plt.tight_layout(); plt.show()

In [ ]:
# 前編 図4　賃金と物価（2020年=100）
ww = w.loc[2013:2025]
fig, ax = plt.subplots(figsize=(10, 5.5))
for c, col, lab in [("price_index", RED, "物価（持家の帰属家賃を除く総合）"), ("nominal_wage_index", NAVY, "名目賃金"), ("real_wage_index", ORANGE, "実質賃金")]:
    ax.plot(ww.index, ww[c], color=col, lw=2.5, label=lab)
ax.axhline(100, color="#999", lw=0.8); ax.set_ylabel("指数（2020年＝100）"); ax.legend(frameon=False)
ax.set_title("図4　賃金は上がったが、物価はもっと上がった", loc="left")
plt.tight_layout(); plt.show()

In [ ]:
# 前編 図5　税収と名目GDP（2012年度＝100）
y5 = list(range(2012, 2026))
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.plot(y5, tax.reindex(y5) / tax[2012] * 100, color=RED, lw=2.5, marker="o", label="一般会計税収")
ax.plot(y5, gdp.reindex(y5) / gdp[2012] * 100, color=NAVY, lw=2.5, marker="o", label="名目GDP")
ax.set_ylabel("指数（2012年度＝100）"); ax.legend(frameon=False)
ax.set_title("図5　税収は、名目GDPよりも速く増えている", loc="left")
plt.tight_layout(); plt.show()
# 図6（所得五分位別インフレ率）は第1部の記事の再現パッケージを参照:
# https://s-v.jp.net/2026/09/16/japan-inflation-inequality-by-income/

In [ ]:
# 後編 図1　第一次所得収支（暦年、兆円）
p = pi.loc[2005:2025]
fig, ax = plt.subplots(figsize=(10, 5.5))
for c, col, lab in [("primary_income", NAVY, "第一次所得収支"), ("direct_investment_income", ORANGE, "うち直接投資収益"), ("reinvested_earnings", GREEN, "うち再投資収益")]:
    ax.plot(p.index, p[c], color=col, lw=2.5, label=lab)
ax.axhline(0, color="#999", lw=0.8); ax.set_ylabel("兆円"); ax.legend(frameon=False)
ax.set_title("図1　海外から受け取る所得は、10年で2倍に", loc="left")
plt.tight_layout(); plt.show()

In [ ]:
# 後編 図2　外貨準備：ドル建てと円換算
d = rs.loc["2012-01-01":]
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(11, 5))
ax.plot(d.index, d["usd_tn"], color=GREEN, lw=2.2); ax.set_ylim(0, 1.6); ax.set_title("ドル建て（兆ドル）", loc="left")
ax2.plot(d.index, d["jpy_tn"], color=NAVY, lw=2.2); ax2.set_ylim(0, 225); ax2.set_title("円に換算した額（兆円）", loc="left")
fig.suptitle("図2　外貨準備：ドルでは横ばい、円で見ると膨らんだ", x=0.01, ha="left")
plt.tight_layout(); plt.show()

## 4. 記事中の、このノートブックで計算していない数値

公表資料の数値をそのまま引用したものです。

- 日銀の国債保有比率 46.7%（2026年6月末速報。国庫短期証券を除き、財投債を含む）：財務省「国債等の保有者別内訳」 https://www.mof.go.jp/jgbs/reference/appendix/holdings01.pdf
- 外為特会の2024年度の運用収入 4.8兆円、剰余金のうち2025年度一般会計への繰入れ 3.2兆円：財務省「令和6年度 外国為替資金特別会計の決算」 https://www.mof.go.jp/about_mof/mof_budget/special_account/gaitame/2024account.html
- 2026年7月30日〜8月26日の為替介入 15兆3,993億円：財務省「外国為替平衡操作の実施状況」 https://www.mof.go.jp/policy/international_policy/reference/feio/index.html
- 所得五分位別インフレ率（下位20% 13.6%、上位20% 11.7%）：SVillage「同じ物価上昇でも、痛みは所得で違った」 https://s-v.jp.net/2026/09/16/japan-inflation-inequality-by-income/